# ASCEND Stage 1 — Held-out re-evaluation of V1

Reruns the existing GRPO pipeline (`ladder_optimizer_v4` + `variant_generator_v2`, unchanged) but evaluates pre/post accuracy on a **held-out** split instead of the training split. This is the first result in the Version 2 continuation — see the project doc for the full audit and roadmap.

Runs on a free Colab T4 GPU. Checkpoints go to Google Drive so a session disconnect doesn't lose progress; the training script itself resumes automatically (`RESUME_INTERACTIVE=0`).

## 1. Mount Drive (for checkpoint persistence across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/ascend_v2_checkpoints'
import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

## 2. Clone your fork (ascend-v2 branch)

In [ ]:
FORK_URL = 'https://github.com/Sithranjan-Suresh/Team_ASCEND.git'

!git clone --branch ascend-v2 $FORK_URL /content/ascend
%cd /content/ascend

# Restore checkpoints from a previous Colab session, if any exist.
!mkdir -p training_runs_holdout
!cp -r "$DRIVE_CHECKPOINT_DIR"/* training_runs_holdout/ 2>/dev/null || echo 'No prior checkpoints found on Drive -- starting fresh.'

## 3. Install dependencies

In [ ]:
# IMPORTANT: do NOT upgrade torch here -- Colab's preinstalled torch is already matched
# to its preinstalled torchvision/torchaudio and CUDA build. Upgrading torch alone (pip -U)
# breaks that binary match (surfaces later as 'RuntimeError: operator torchvision::nms
# does not exist' when transformers/peft import). Only install what Colab lacks.
!pip install -q transformers datasets accelerate peft trl bitsandbytes

## 4. Build the stratified train/held-out split

Stratified by P3's `module` field (18 puzzle categories). Fixed seed → reproducible split, and it's saved into the repo so every later experiment (E1–E8 in the doc) can reuse the exact same held-out set.

In [ ]:
!python split_dataset.py --seed 42 --heldout-frac 0.2

## 5. Run training + held-out evaluation

`TRAIN_LIMIT` / `HELDOUT_EVAL_LIMIT` default to 100 each, matching V1's `main_v2.py` scale. Lower these first if you just want a quick smoke test (e.g. 10/10) before committing a full T4 session to it.

In [ ]:
import os
os.environ['SPLIT_SEED'] = '42'
os.environ['TRAIN_LIMIT'] = '100'
os.environ['HELDOUT_EVAL_LIMIT'] = '100'
os.environ['RESUME_INTERACTIVE'] = '0'

!python run_holdout_eval.py

## 6. Back up checkpoints to Drive

Run this periodically during a long training loop (Colab can disconnect at any time), and again at the end.

In [ ]:
!cp -r training_runs_holdout/* "$DRIVE_CHECKPOINT_DIR"/
print('Checkpoints backed up to', DRIVE_CHECKPOINT_DIR)

## 7. Read back the result

Compare `pretrain_holdout` / `posttrain_holdout` here against the training-set numbers in the original `result*.txt` files on `integration-v1`. A much smaller (or vanished) gap on held-out data means V1's reported pass@k improvement was largely training-set memorization.

In [ ]:
import json, glob
run_dirs = sorted(glob.glob('training_runs_holdout/*/run.json'))
for rd in run_dirs:
    with open(rd) as f:
        data = json.load(f)
    print(rd)
    print('  status:', data.get('status'))
    print('  pretrain_holdout: ', data.get('results', {}).get('pretrain_holdout'))
    print('  posttrain_holdout:', data.get('results', {}).get('posttrain_holdout'))

## 8. (Optional) Push the split file + results back to your fork

Uses a GitHub personal access token, entered here and never written to disk or the repo — generate one at https://github.com/settings/tokens (repo scope) if you don't have one.

In [ ]:
import getpass
token = getpass.getpass('GitHub personal access token: ')

!git config user.email "you@example.com"
!git config user.name "Sithranjan Suresh"
!git add PythonProgrammingPuzzles/puzzles/split_seed42.json
!git commit -m "Stage 1: add stratified train/held-out split (seed 42)" || echo 'nothing to commit'
!git push https://$token@github.com/Sithranjan-Suresh/Team_ASCEND.git ascend-v2

del token